# CLASSIFIER ABLATION STUDY

In [16]:
import pickle
#load from saved pkl
with open("cache/dpf_mvsa_single_results.pkl", "rb") as f:
    results = pickle.load(f)

# RETRIEVE  Varibles from results
train_probs_text   = results["train_probs_text"]
train_probs_image  = results["train_probs_image"]

val_probs_text     = results["val_probs_text"]
val_probs_image    = results["val_probs_image"]

test_probs_text    = results["test_probs_text"]
test_probs_image   = results["test_probs_image"]

y_train_int = results["y_train_int"]
y_val_int   = results["y_val_int"]
y_test_int  = results["y_test_int"]

beta  = results["beta"]
topk  = results["topk"]
delta = results["delta"]

In [17]:
import numpy as np
import xgboost as xgb
import time
from sklearn.ensemble import ExtraTreesClassifier
import cs16.DPF as DPF #load Private Library CS16
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# # >> Global Fusion：Combine Text(3) + Image(3) -> 6 Dimensions
########################################
beta = 1.1
topk=2
delta =1e-2

# 4. Dynamic Fusion
X_train_global = DPF.enhanced_dynamic_fusion_topk_adaptive(train_probs_text, train_probs_image, beta, topk, delta)
X_val_global   = DPF.enhanced_dynamic_fusion_topk_adaptive(val_probs_text,   val_probs_image, beta, topk, delta)
X_test_global  = DPF.enhanced_dynamic_fusion_topk_adaptive(test_probs_text,  test_probs_image, beta, topk, delta)
print("Dynamic Fusion Feature shape (train):", X_train_global.shape)
# 5. Global Classifier
global_clf = xgb.XGBClassifier(n_estimators=350, max_depth=5, random_state=42)
global_clf.fit(X_train_global, y_train_int)

# test
test_pred = global_clf.predict(X_test_global)
print("Test Accuracy:", accuracy_score(y_test_int, test_pred))
print("Test Classification Report:")
print(classification_report(y_test_int, test_pred,
                            digits=4,
                            target_names=['negative','neutral','positive']))


Dynamic Fusion Feature shape (train): (3895, 6)
Test Accuracy: 0.5708418891170431
Test Classification Report:
              precision    recall  f1-score   support

    negative     0.3793    0.1429    0.2075        77
     neutral     0.6350    0.7698    0.6959       278
    positive     0.4380    0.4015    0.4190       132

    accuracy                         0.5708       487
   macro avg     0.4841    0.4381    0.4408       487
weighted avg     0.5412    0.5708    0.5436       487



<a id="classifier"></a>
# 📈 META-CLASSIFIER ABLATION STUDY 
### (5 runs with different seeds)
[↑ Back to Top](#top)

In [18]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb
from sklearn.metrics import f1_score, accuracy_score

# ============================================================
# Meta-classifier Ablation Study with Error Bars
# ============================================================

# Define seeds for multiple runs
seeds = [42, 123, 456, 789, 101112]  # 5 runs, can increase to 10

# Define classifiers to compare
meta_classifiers = {
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial'),
    'SVM': SVC(kernel='rbf', random_state=42, probability=True),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'MLP': MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=300, max_depth=5, random_state=42, eval_metric='mlogloss')
}

# Store results
results_meta = {}

print("="*70)
print("META-CLASSIFIER ABLATION STUDY (5 runs with different seeds)")
print("="*70)

for name, clf in meta_classifiers.items():
    f1_scores = []
    acc_scores = []
    
    print(f"\n--- {name} ---")
    
    for seed in seeds:
        # Set random seed for reproducibility
        if hasattr(clf, 'random_state'):
            clf.set_params(random_state=seed)
        
        # Train and predict
        clf.fit(X_train_global, y_train_int)
        pred = clf.predict(X_test_global)
        
        # Calculate metrics
        f1 = f1_score(y_test_int, pred, average='weighted')
        acc = accuracy_score(y_test_int, pred)
        
        f1_scores.append(f1)
        acc_scores.append(acc)
        
        print(f"  Seed {seed}: F1={f1:.4f}, Acc={acc:.4f}")
    
    # Calculate statistics
    f1_mean = np.mean(f1_scores)
    f1_std = np.std(f1_scores)
    acc_mean = np.mean(acc_scores)
    acc_std = np.std(acc_scores)
    
    results_meta[name] = {
        'f1_mean': f1_mean,
        'f1_std': f1_std,
        'acc_mean': acc_mean,
        'acc_std': acc_std,
        'f1_scores': f1_scores,
        'acc_scores': acc_scores
    }
    
    print(f"  → Mean ± Std: F1={f1_mean:.4f}±{f1_std:.4f}, Acc={acc_mean:.4f}±{acc_std:.4f}")

# ============================================================
# Print summary table
# ============================================================

print("\n" + "="*70)
print("SUMMARY TABLE (Mean ± Std over 5 runs)")
print("="*70)
print(f"{'Meta-classifier':<20} {'Weighted F1':<20} {'Accuracy':<20}")
print("-"*60)

for name in results_meta:
    f1_mean = results_meta[name]['f1_mean']
    f1_std = results_meta[name]['f1_std']
    acc_mean = results_meta[name]['acc_mean']
    acc_std = results_meta[name]['acc_std']
    
    print(f"{name:<20} {f1_mean:.4f} ± {f1_std:.4f}     {acc_mean:.4f} ± {acc_std:.4f}")

# ============================================================
# Statistical test: XGBoost vs second best
# ============================================================

print("\n" + "="*70)
print("STATISTICAL COMPARISON (Wilcoxon signed-rank test)")
print("="*70)

# Find second best by F1 mean
sorted_names = sorted(results_meta.keys(), key=lambda x: results_meta[x]['f1_mean'], reverse=True)
best_name = sorted_names[0]
second_name = sorted_names[1]

from scipy.stats import wilcoxon

best_scores = results_meta[best_name]['f1_scores']
second_scores = results_meta[second_name]['f1_scores']

_, p_value = wilcoxon(best_scores, second_scores)
print(f"{best_name} vs {second_name}: p = {p_value:.4f}")

if p_value < 0.05:
    print(f"✓ {best_name} significantly outperforms {second_name} (p < 0.05)")
else:
    print(f"⚠ No significant difference between {best_name} and {second_name}")

# ============================================================
# Rebuttal-ready table
# ============================================================

print("\n" + "="*70)
print("SUMMARY TABLE")
print("="*70)
print()
print("| Classifier          | w.F1(mean ± std)| Acc (mean ± std)|")
print("|---------------------|-----------------|-----------------|")

for name in sorted_names:
    f1_mean = results_meta[name]['f1_mean']
    f1_std = results_meta[name]['f1_std']
    acc_mean = results_meta[name]['acc_mean']
    acc_std = results_meta[name]['acc_std']
    
    #print(f"| {name}       | {f1_mean:.4f} ± {f1_std:.4f} | {acc_mean:.4f} ± {acc_std:.4f} |")
    print(f"| {name:<19} | {f1_mean:.4f} ± {f1_std:.4f} | {acc_mean:.4f} ± {acc_std:.4f} |")

META-CLASSIFIER ABLATION STUDY (5 runs with different seeds)

--- Random Forest ---
  Seed 42: F1=0.5229, Acc=0.5626
  Seed 123: F1=0.5258, Acc=0.5647
  Seed 456: F1=0.5275, Acc=0.5708
  Seed 789: F1=0.5229, Acc=0.5647
  Seed 101112: F1=0.5094, Acc=0.5565
  → Mean ± Std: F1=0.5217±0.0064, Acc=0.5639±0.0046

--- Logistic Regression ---
  Seed 42: F1=0.4849, Acc=0.5544
  Seed 123: F1=0.4849, Acc=0.5544
  Seed 456: F1=0.4849, Acc=0.5544
  Seed 789: F1=0.4849, Acc=0.5544
  Seed 101112: F1=0.4849, Acc=0.5544
  → Mean ± Std: F1=0.4849±0.0000, Acc=0.5544±0.0000

--- SVM ---
  Seed 42: F1=0.4911, Acc=0.5236
  Seed 123: F1=0.4911, Acc=0.5236
  Seed 456: F1=0.4911, Acc=0.5236
  Seed 789: F1=0.4911, Acc=0.5236
  Seed 101112: F1=0.4911, Acc=0.5236
  → Mean ± Std: F1=0.4911±0.0000, Acc=0.5236±0.0000

--- KNN ---
  Seed 42: F1=0.5213, Acc=0.5462
  Seed 123: F1=0.5213, Acc=0.5462
  Seed 456: F1=0.5213, Acc=0.5462
  Seed 789: F1=0.5213, Acc=0.5462
  Seed 101112: F1=0.5213, Acc=0.5462
  → Mean ± Std: F

# Beta = -1 

As an auxiliary diagnostic, we also examined negative β, which reverses the ITP-induced weighting direction. It produced no statistically reliable improvement over the uniform mode on CH-SIMS and was therefore excluded from the valid DPF operating space.

# MOSI Full Upstream Sensitivity & DPF Activation

# Activation

In [14]:
"""
CH-SIMS: Activation Analysis - When Does DPF Activate?
Properly uses Validation for Beta Selection + OOF Probabilities
"""

import pickle
import time
import numpy as np
import xgboost as xgb
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score
import cs16.DPF as DPF

start_time = time.time()

# ============================================================
# 1. Load CH-SIMS Data
# ============================================================
file_path = 'SIMS/Processed/unaligned_39.pkl'
with open(file_path, 'rb') as f:
    data = pickle.load(f)

train = data['train']
valid = data['valid']
test = data['test']

def aggregate_mean(X):
    return np.mean(X, axis=1)

X_text_train = aggregate_mean(train['text'])
X_vision_train = aggregate_mean(train['vision'])
y_train = train['classification_labels'].astype(int)

X_text_val = aggregate_mean(valid['text'])
X_vision_val = aggregate_mean(valid['vision'])
y_val = valid['classification_labels'].astype(int)

X_text_test = aggregate_mean(test['text'])
X_vision_test = aggregate_mean(test['vision'])
y_test = test['classification_labels'].astype(int)

# ============================================================
# 2. Convert to Binary Classification
# ============================================================
def convert_to_binary(X_text, X_vision, y):
    mask = (y != 1)
    X_text_bin = X_text[mask]
    X_vision_bin = X_vision[mask]
    y_bin = y[mask]
    y_bin = np.where(y_bin == 2, 1, 0)
    return X_text_bin, X_vision_bin, y_bin

X_text_train_bin, X_vision_train_bin, y_train_bin = convert_to_binary(
    X_text_train, X_vision_train, y_train
)
X_text_val_bin, X_vision_val_bin, y_val_bin = convert_to_binary(
    X_text_val, X_vision_val, y_val
)
X_text_test_bin, X_vision_test_bin, y_test_bin = convert_to_binary(
    X_text_test, X_vision_test, y_test
)

# Combine train + val for OOF (standard practice)
X_text_all = np.vstack([X_text_train_bin, X_text_val_bin])
X_vision_all = np.vstack([X_vision_train_bin, X_vision_val_bin])
y_all = np.concatenate([y_train_bin, y_val_bin])

print("=" * 60)
print("CH-SIMS: Activation Analysis (Validation Selection + OOF)")
print("=" * 60)
print(f"Train: {len(y_train_bin)}, Val: {len(y_val_bin)}, Test: {len(y_test_bin)}")
print(f"Labels (train): {np.bincount(y_train_bin)}")
print("=" * 60)

# ============================================================
# 3. Generate OOF Probabilities (Leakage-Free)
# ============================================================
print("\nGenerating OOF probabilities...")

SEED = 42
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_splits = list(skf.split(X_text_all, y_all))

def generate_oof_probs(X, y, cv_splits, estimator_class, **kwargs):
    """Generate OOF probabilities for upstream models."""
    n_samples = len(X)
    n_classes = len(np.unique(y))
    oof_probs = np.zeros((n_samples, n_classes))
    
    for train_idx, val_idx in cv_splits:
        # Split
        X_train_fold = X[train_idx]
        y_train_fold = y[train_idx]
        X_val_fold = X[val_idx]
        
        # Train
        clf = estimator_class(**kwargs)
        clf.fit(X_train_fold, y_train_fold)
        
        # Predict OOF
        oof_probs[val_idx] = clf.predict_proba(X_val_fold)
    
    return oof_probs

# Text OOF
text_kwargs = {
    'n_estimators': 100,
    'class_weight': 'balanced',
    'random_state': SEED
}
text_oof = generate_oof_probs(X_text_all, y_all, cv_splits, ExtraTreesClassifier, **text_kwargs)

# Vision OOF
vision_kwargs = {
    'n_estimators': 100,
    'max_depth': 12,
    'min_samples_leaf': 6,
    'max_features': 'sqrt',
    'class_weight': 'balanced',
    'bootstrap': True,
    'random_state': SEED
}
vision_oof = generate_oof_probs(X_vision_all, y_all, cv_splits, ExtraTreesClassifier, **vision_kwargs)

# Split OOF back to train and val
text_oof_train = text_oof[:len(y_train_bin)]
text_oof_val = text_oof[len(y_train_bin):]
vision_oof_train = vision_oof[:len(y_train_bin)]
vision_oof_val = vision_oof[len(y_train_bin):]

print(f"Text OOF shape: {text_oof.shape}")
print(f"Vision OOF shape: {vision_oof.shape}")

# ============================================================
# 4. Generate Test Probabilities (Full Model)
# ============================================================
print("\nTraining full upstream models for test...")

clf_text = ExtraTreesClassifier(**text_kwargs)
clf_text.fit(X_text_all, y_all)
test_probs_text = clf_text.predict_proba(X_text_test_bin)

clf_vision = ExtraTreesClassifier(**vision_kwargs)
clf_vision.fit(X_vision_all, y_all)
test_probs_vision = clf_vision.predict_proba(X_vision_test_bin)

print(f"Test Text probs: {test_probs_text.shape}")
print(f"Test Vision probs: {test_probs_vision.shape}")

# ============================================================
# 5. Activation Analysis: Beta Selection on Validation
# ============================================================
print("\n" + "=" * 60)
print("ACTIVATION ANALYSIS: Beta Selection on Validation")
print("=" * 60)

TOPK = 2
DELTA = 1e-2
BETA_CANDIDATES = [0,0.5,1,2,3,5]

def evaluate_beta(beta, train_text, train_vision, val_text, val_vision,
                  test_text, test_vision, y_train, y_val, y_test):
    """Evaluate a single beta value."""
    # Fuse
    X_train = DPF.enhanced_dynamic_fusion_topk_adaptive(
        train_text, train_vision, beta, TOPK, DELTA
    )
    X_val = DPF.enhanced_dynamic_fusion_topk_adaptive(
        val_text, val_vision, beta, TOPK, DELTA
    )
    X_test = DPF.enhanced_dynamic_fusion_topk_adaptive(
        test_text, test_vision, beta, TOPK, DELTA
    )
    
    # Train downstream on OOF training
    clf = xgb.XGBClassifier(
        n_estimators=350,
        max_depth=5,
        random_state=SEED,
        eval_metric='logloss',
        use_label_encoder=False,
        verbosity=0
    )
    clf.fit(X_train, y_train)
    
    # Validation (for beta selection)
    val_pred = clf.predict(X_val)
    val_f1 = f1_score(y_val, val_pred, average='weighted')
    
    # Test (for final report only)
    test_pred = clf.predict(X_test)
    test_acc = accuracy_score(y_test, test_pred)
    test_f1 = f1_score(y_test, test_pred, average='weighted')
    
    return {
        'beta': beta,
        'val_f1': val_f1,
        'test_acc': test_acc,
        'test_f1': test_f1
    }

# Evaluate all beta candidates
results = []
for beta in BETA_CANDIDATES:
    r = evaluate_beta(
        beta,
        text_oof_train, vision_oof_train,
        text_oof_val, vision_oof_val,
        test_probs_text, test_probs_vision,
        y_train_bin, y_val_bin, y_test_bin
    )
    results.append(r)
    print(f"β={beta:.2f}: Val F1={r['val_f1']:.4f}, Test F1={r['test_f1']:.4f}")

# ============================================================
# 6. Select Best Beta by Validation (Tie-break: smaller beta)
# ============================================================
best_by_val = max(results, key=lambda x: (x['val_f1'], -x['beta']))
selected_beta = best_by_val['beta']
selected_result = next(r for r in results if r['beta'] == selected_beta)
uniform_result = next(r for r in results if r['beta'] == 0.0)

# ============================================================
# 7. Activation Summary
# ============================================================
print("\n" + "=" * 60)
print("ACTIVATION SUMMARY (Beta Selected by Validation)")
print("=" * 60)

print(f"\nSelected β* = {selected_beta:.2f} (Validation F1 = {best_by_val['val_f1']:.4f})")
print(f"Uniform β=0  : Val F1 = {uniform_result['val_f1']:.4f}")

# Improvement on Validation
val_improvement = best_by_val['val_f1'] - uniform_result['val_f1']
print(f"Validation Δ: +{val_improvement:.4f} F1")

# Improvement on Test (for reporting only)
test_improvement = selected_result['test_f1'] - uniform_result['test_f1']
print(f"\nUniform Test F1: {uniform_result['test_f1']:.4f}")
print(f"Selected Test F1: {selected_result['test_f1']:.4f}")
print(f"Test Δ: {test_improvement:+.4f} F1")

# ============================================================
# 8. Activation Status (Precise Definition)
# ============================================================
print("\n" + "=" * 60)
print("ACTIVATION STATUS")
print("=" * 60)

# Condition 1: Validation selected adaptive weighting
if selected_beta > 0:
    print("✅ β* > 0: Validation selected adaptive weighting (DPF activated)")
else:
    print("❌ β* = 0: Validation selected uniform fallback (DPF not activated)")

# Condition 2: Test improvement
if test_improvement > 0:
    print(f"✅ Test Δ > 0: Activation produces improvement on test (+{test_improvement:.4f} F1)")
else:
    print(f"❌ Test Δ ≤ 0: Activation does not improve test performance ({test_improvement:+.4f} F1)")

print("\n" + "-" * 60)
if selected_beta > 0 and test_improvement > 0:
    print("✅ DPF SUCCESSFULLY ACTIVATED: Validation selected β>0 AND test improves.")
elif selected_beta > 0 and test_improvement <= 0:
    print("⚠️ DPF ACTIVATED BUT OVERFIT: Validation selected β>0 but test does not improve.")
else:
    print("ℹ️ DPF NOT ACTIVATED: Validation selected uniform fusion.")

# ============================================================
# 9. Detailed Results Table
# ============================================================
print("\n" + "=" * 60)
print("DETAILED BETA SCAN (Validation Selection)")
print("=" * 60)
print(f"{'Beta':<8} {'Val F1':<12} {'Test F1':<12} {'Selected':<12}")
print("-" * 48)

for r in results:
    is_selected = "✅" if r['beta'] == selected_beta else ""
    print(f"{r['beta']:<8.2f} {r['val_f1']:<12.4f} {r['test_f1']:<12.4f} {is_selected:<12}")

# ============================================================
# 10. Runtime
# ============================================================
end_time = time.time()
elapsed_time = end_time - start_time
print(f"\nTotal Execution Time: {elapsed_time:.4f} seconds")

CH-SIMS: Activation Analysis (Validation Selection + OOF)
Train: 1161, Val: 387, Test: 388
Labels (train): [742 419]

Generating OOF probabilities...
Text OOF shape: (1548, 2)
Vision OOF shape: (1548, 2)

Training full upstream models for test...
Test Text probs: (388, 2)
Test Vision probs: (388, 2)

ACTIVATION ANALYSIS: Beta Selection on Validation


C:\Users\ausco\anaconda3\lib\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")


β=0.00: Val F1=0.7021, Test F1=0.7391


C:\Users\ausco\anaconda3\lib\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")


β=0.50: Val F1=0.7063, Test F1=0.7355


...

β=2.00: Val F1=0.6881, Test F1=0.7330


C:\Users\ausco\anaconda3\lib\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")


β=3.00: Val F1=0.6777, Test F1=0.7276


C:\Users\ausco\anaconda3\lib\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")


β=5.00: Val F1=0.7194, Test F1=0.7468

ACTIVATION SUMMARY (Beta Selected by Validation)

Selected β* = 5.00 (Validation F1 = 0.7194)
Uniform β=0  : Val F1 = 0.7021
Validation Δ: +0.0173 F1

Uniform Test F1: 0.7391
Selected Test F1: 0.7468
Test Δ: +0.0077 F1

ACTIVATION STATUS
✅ β* > 0: Validation selected adaptive weighting (DPF activated)
✅ Test Δ > 0: Activation produces improvement on test (+0.0077 F1)

------------------------------------------------------------
✅ DPF SUCCESSFULLY ACTIVATED: Validation selected β>0 AND test improves.

DETAILED BETA SCAN (Validation Selection)
Beta     Val F1       Test F1      Selected    
------------------------------------------------
0.00     0.7021       0.7391                   
0.50     0.7063       0.7355                   
1.00     0.7012       0.7502                   
2.00     0.6881       0.7330                   
3.00     0.6777       0.7276                   
5.00     0.7194       0.7468       ✅           

Total Execution Time: 7.528

# Information-Energy Formulation (Rényi-2 vs. Shannon)
While the unregu-larized ITP is mathematically Rényi-2, DPF is designed from aninformation-energy perspective—not as an entropy-based weight-ing.

In [6]:
import pickle
import numpy as np
import xgboost as xgb
from sklearn.metrics import f1_score
from scipy.special import softmax
from itertools import product

# ============================================================
# 1. LOAD DATA
# ============================================================
with open("cache/dpf_mvsa_single_results.pkl", "rb") as f:
    results = pickle.load(f)

train_probs_text  = results["train_probs_text"]
train_probs_image = results["train_probs_image"]
val_probs_text    = results["val_probs_text"]
val_probs_image   = results["val_probs_image"]
test_probs_text   = results["test_probs_text"]
test_probs_image  = results["test_probs_image"]
y_train_int = results["y_train_int"]
y_val_int   = results["y_val_int"]
y_test_int  = results["y_test_int"]

# ============================================================
# 2. FUNCTION DEFINITIONS
# ============================================================

def compute_shannon(probs, eps=1e-12):
    """Shannon entropy (positive, lower = more concentrated)"""
    return -np.sum(probs * np.log(probs + eps), axis=-1)

def compute_renyi2(probs):
    """Rényi-2 entropy (full distribution, no regularization)"""
    return -np.log(np.sum(probs**2, axis=-1))

def compute_practical_itp(probs, topk=2, delta=1e-2):
    """Practical ITP from DPF (with top-k and regularization)"""
    C = probs.shape[-1]
    topk_indices = np.argsort(-probs, axis=-1)[:, :topk]
    sum_sq = np.sum(np.take_along_axis(probs, topk_indices, axis=-1)**2, axis=-1)
    eps = delta * np.max(np.take_along_axis(probs, topk_indices, axis=-1)**2, axis=-1)
    return -np.log(sum_sq + eps)

def compute_gibbs_weights(itp_per_sample, beta):
    """Gibbs-style weighting from ITP values"""
    return softmax(-beta * itp_per_sample, axis=-1)

def evaluate_beta(probs_text, probs_image, y_train, y_val, y_test, beta, itp_func, itp_kwargs=None):
    """Evaluate a given beta and ITP function"""
    if itp_kwargs is None:
        itp_kwargs = {}
    
    # Compute ITPs for text and image
    itp_text_train = itp_func(probs_text[0], **itp_kwargs)
    itp_image_train = itp_func(probs_image[0], **itp_kwargs)
    itp_text_val = itp_func(probs_text[1], **itp_kwargs)
    itp_image_val = itp_func(probs_image[1], **itp_kwargs)
    itp_text_test = itp_func(probs_text[2], **itp_kwargs)
    itp_image_test = itp_func(probs_image[2], **itp_kwargs)
    
    # Stack ITPs: (n_samples, 2)
    itp_train = np.stack([itp_text_train, itp_image_train], axis=-1)
    itp_val = np.stack([itp_text_val, itp_image_val], axis=-1)
    itp_test = np.stack([itp_text_test, itp_image_test], axis=-1)
    
    # Gibbs weights
    w_train = compute_gibbs_weights(itp_train, beta)
    w_val = compute_gibbs_weights(itp_val, beta)
    w_test = compute_gibbs_weights(itp_test, beta)
    
    # Weighted probabilities (element-wise multiplication)
    p_train_weighted = np.stack([
        w_train[:, 0:1] * probs_text[0],
        w_train[:, 1:2] * probs_image[0]
    ], axis=-1)  # (n, C, 2)
    p_val_weighted = np.stack([
        w_val[:, 0:1] * probs_text[1],
        w_val[:, 1:2] * probs_image[1]
    ], axis=-1)
    p_test_weighted = np.stack([
        w_test[:, 0:1] * probs_text[2],
        w_test[:, 1:2] * probs_image[2]
    ], axis=-1)
    
    # Flatten for classifier
    X_train_flat = p_train_weighted.reshape(p_train_weighted.shape[0], -1)
    X_val_flat = p_val_weighted.reshape(p_val_weighted.shape[0], -1)
    X_test_flat = p_test_weighted.reshape(p_test_weighted.shape[0], -1)
    
    # Train XGBoost on training set
    model = xgb.XGBClassifier(
        n_estimators=350,
        max_depth=5,
        learning_rate=0.1,
        random_state=42,
        eval_metric='mlogloss'
    )
    model.fit(X_train_flat, y_train)
    
    # Evaluate on validation and test
    y_val_pred = model.predict(X_val_flat)
    y_test_pred = model.predict(X_test_flat)
    
    val_f1 = f1_score(y_val, y_val_pred, average='weighted')
    test_f1 = f1_score(y_test, y_test_pred, average='weighted')
    
    return {'val_f1': val_f1, 'test_f1': test_f1}

# ============================================================
# 3. SCAN BETA FOR EACH ITP FUNCTION
# ============================================================

probs_text = [train_probs_text, val_probs_text, test_probs_text]
probs_image = [train_probs_image, val_probs_image, test_probs_image]

# Beta search ranges
beta_list_shannon = [0.1, 0.5, 1.0, 2.0, 4.0, 6.0, 8.0, 10.0, 15.0]
beta_list_renyi2 = [0.1, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0, 8.0]
beta_list_practical = [0.1, 0.5, 1.0, 1.1, 2.0, 3.0, 4.0, 5.0]

def scan_beta(probs_text, probs_image, y_train, y_val, y_test, beta_list, itp_func, itp_kwargs=None):
    results = []
    for beta in beta_list:
        print(f"  beta={beta:.2f}")
        res = evaluate_beta(
            probs_text, probs_image, y_train, y_val, y_test,
            beta, itp_func, itp_kwargs
        )
        results.append({
            'beta': beta,
            'val_f1': res['val_f1'],
            'test_f1': res['test_f1']
        })
    return results

# ============================================================
# 4. RUN EXPERIMENTS
# ============================================================

print("=" * 60)
print("Experiment 1: Shannon Entropy")
print("=" * 60)
shannon_results = scan_beta(
    probs_text, probs_image, y_train_int, y_val_int, y_test_int,
    beta_list_shannon, compute_shannon
)

print("\n" + "=" * 60)
print("Experiment 2: Rényi-2 Entropy")
print("=" * 60)
renyi2_results = scan_beta(
    probs_text, probs_image, y_train_int, y_val_int, y_test_int,
    beta_list_renyi2, compute_renyi2
)

print("\n" + "=" * 60)
print("Experiment 3: Practical ITP (DPF)")
print("=" * 60)
practical_results = scan_beta(
    probs_text, probs_image, y_train_int, y_val_int, y_test_int,
    beta_list_practical, compute_practical_itp, {'topk': 2, 'delta': 1e-2}
)

# ============================================================
# 5. PRINT SUMMARY
# ============================================================

def print_summary(results, name):
    best = max(results, key=lambda x: x['val_f1'])
    print(f"\n{name}:")
    print(f"  Best beta = {best['beta']:.2f}")
    print(f"  Val F1    = {best['val_f1']:.4f}")
    print(f"  Test F1   = {best['test_f1']:.4f}")

print_summary(shannon_results, "Shannon Entropy")
print_summary(renyi2_results, "Rényi-2 Entropy")
print_summary(practical_results, "Practical ITP")

# ============================================================
# 6. SAVE RESULTS
# ============================================================

all_results = {
    'shannon': shannon_results,
    'renyi2': renyi2_results,
    'practical_itp': practical_results
}


Experiment 1: Shannon Entropy
  beta=0.10
  beta=0.50
  beta=1.00
  beta=2.00
  beta=4.00
  beta=6.00
  beta=8.00
  beta=10.00
  beta=15.00

Experiment 2: Rényi-2 Entropy
  beta=0.10
  beta=0.50
  beta=1.00
  beta=1.50
  beta=2.00
  beta=3.00
  beta=4.00
  beta=5.00
  beta=6.00
  beta=8.00

Experiment 3: Practical ITP (DPF)
  beta=0.10
  beta=0.50
  beta=1.00
  beta=1.10
  beta=2.00
  beta=3.00
  beta=4.00
  beta=5.00

Shannon Entropy:
  Best beta = 2.00
  Val F1    = 0.4861
  Test F1   = 0.5148

Rényi-2 Entropy:
  Best beta = 0.50
  Val F1    = 0.4774
  Test F1   = 0.5182

Practical ITP:
  Best beta = 1.10
  Val F1    = 0.4777
  Test F1   = 0.5224
